# EDJ Interaction — Velocity & Richardson Number at 1°N, 0°N, 1°S, 140°W

For each of 3 simulations (3month, 3month_Ri3, 3month_Ri5), produces 6 figures:
- 3 locations (1°N, 0°N, 1°S) × 2 time ranges (full period, first 4 weeks)

Each figure: **6 rows × 2 columns** (left = UVEL, right = VVEL), upper 1200 m
1. Raw velocity
2. Shear: dU/dz (left), dV/dz (right)
3. Ri = N²/S², S² = (dU/dz)² + (dV/dz)² — same field in both columns, `RdYlGn` colormap with `TwoSlopeNorm` centred at the simulation's Ri threshold
4. Band-passed 40–60 days
5. Band-passed 25–40 days
6. Band-passed 15–25 days

Requires `cache_edj.nc` in each subfolder — run `build_edj_cache.py` first.

In [1]:
import numpy as np
import xarray as xr
import pandas as pd
import os
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
import cmocean.cm as cmo
import warnings
from scipy.signal import butter, filtfilt

warnings.filterwarnings('ignore')
plt.rcParams.update({'font.size': 10, 'axes.titlesize': 10, 'axes.labelsize': 9})

# Notebook directory (works with nbconvert --inplace executed from this directory)
NOTEBOOK_DIR = os.path.abspath('')

SUBFOLDERS    = ['3month', '3month_Ri3', '3month_Ri5']
RI_THRESHOLDS = {'3month': 0.7, '3month_Ri3': 0.3, '3month_Ri5': 0.5}
LAT_LABELS    = ['1N', '0N', '1S']
LAT_STR       = {'1N': '1°N', '0N': '0°N', '1S': '1°S'}

G    = 9.806   # m s⁻²
RHO0 = 1025.0  # kg m⁻³

BANDS = [
    ('40–60 day', (40 * 24., 60 * 24.)),
    ('25–40 day', (25 * 24., 40 * 24.)),
    ('15–25 day', (15 * 24., 25 * 24.)),
]

print('NOTEBOOK_DIR:', NOTEBOOK_DIR)

NOTEBOOK_DIR: /home/edavenport/analysis/tpose24-validation/EDJ_interaction


## Load cache

In [2]:
def load_cache(subfolder):
    """Load EDJ cache for a simulation subfolder."""
    path = os.path.join(NOTEBOOK_DIR, subfolder, 'cache_edj.nc')
    if not os.path.exists(path):
        raise FileNotFoundError(
            f'Cache not found: {path}\n'
            f'Run: conda run -n tpose python build_edj_cache.py {subfolder}')
    ds    = xr.open_dataset(path)
    Z_mod = ds['depth'].values          # negative, e.g. -0.5 to -1250 m
    t_mod = pd.DatetimeIndex(ds['time'].values)
    data  = {loc: {'U':      ds[f'U_{loc}'].values,
                   'V':      ds[f'V_{loc}'].values,
                   'DRHODR': ds[f'DRHODR_{loc}'].values}
             for loc in ['0N', '1N', '1S']}
    ds.close()
    print(f'  Loaded {subfolder}: {len(t_mod)} steps, '
          f'{t_mod[0].date()} – {t_mod[-1].date()}, '
          f'{len(Z_mod)} levels ({Z_mod[0]:.1f} to {Z_mod[-1]:.1f} m)')
    return data, Z_mod, t_mod

## Band-pass filter

In [3]:
def bpfilt(data_2d, dt_h, band_periods_h, order=3):
    """
    Zero-phase Butterworth filter applied column-by-column (each depth).
    band_periods_h: (T_lo, T_hi) in hours for band-pass.
    Uses a lenient min_pts so that 40-60 day bands are attempted on 3-month data.
    """
    fs  = 1.0 / dt_h
    nyq = fs / 2.0
    T_lo, T_hi = band_periods_h
    Wn = [max((1.0 / T_hi) / nyq, 1e-6),
          min((1.0 / T_lo) / nyq, 0.99)]
    b, a = butter(order, Wn, btype='band')

    out   = np.full_like(data_2d, np.nan)
    t_idx = np.arange(data_2d.shape[0])

    for iz in range(data_2d.shape[1]):
        col = data_2d[:, iz]
        ok  = np.isfinite(col)
        if ok.sum() < 50:
            continue
        col_f = np.interp(t_idx, t_idx[ok], col[ok])
        col_f -= col_f.mean()
        try:
            filt = filtfilt(b, a, col_f)
        except Exception:
            continue
        filt[~ok] = np.nan
        out[:, iz] = filt
    return out

## Richardson number

In [4]:
def compute_Ri(U, V, DRHODR, Z):
    """
    Ri = N² / S²
    N² = -(g/rho0) * DRHODR, clipped to >= 0
    S² = (dU/dz)² + (dV/dz)²
    Returns Ri array of shape (time, depth).
    """
    N2   = -(G / RHO0) * DRHODR
    N2   = np.maximum(N2, 0.0)
    dUdz = np.gradient(U, Z, axis=1)
    dVdz = np.gradient(V, Z, axis=1)
    S2   = dUdz ** 2 + dVdz ** 2
    return np.where(S2 > 1e-12, N2 / S2, np.nan)

## Plotting helpers

In [5]:
def depth_yax(ax):
    ax.set_ylim([-1200, 0])
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{int(-y)}'))
    ax.set_ylabel('Depth (m)')


def fmt_time_ax(ax, day_interval=None):
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
    if day_interval:
        ax.xaxis.set_major_locator(mdates.DayLocator(interval=day_interval))
    else:
        ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0, interval=2))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')


def vmax_of(*arrays):
    vals = np.concatenate([a[np.isfinite(a)].ravel() for a in arrays])
    if vals.size == 0:
        return 0.1
    return max(float(np.nanpercentile(np.abs(vals), 97)), 0.005)


def cf_plot(ax, t_arr, Z, data, vmax, n_levels=101):
    levels = np.linspace(-vmax, vmax, n_levels)
    return ax.contourf(t_arr, Z, data.T, levels=levels,
                       cmap=cmo.balance, extend='both')


def ri_plot(ax, t_arr, Z, Ri, threshold, n_levels=100):
    """Contourf of Ri with TwoSlopeNorm centred at threshold."""
    vmax  = 5.0 * threshold
    Ri_c  = np.clip(Ri, 0.0, vmax)
    norm  = mcolors.TwoSlopeNorm(vmin=0.0, vcenter=threshold, vmax=vmax)
    lev_lo = np.linspace(0.0, threshold,  n_levels // 2 + 1)
    lev_hi = np.linspace(threshold, vmax, n_levels // 2 + 1)[1:]
    levels = np.concatenate([lev_lo, lev_hi])
    return ax.contourf(t_arr, Z, Ri_c.T, levels=levels,
                       cmap='RdYlGn', norm=norm, extend='max')


def add_vel_cbar(fig, cf, ax_pair):
    cb = fig.colorbar(cf, ax=ax_pair, shrink=0.85, pad=0.01)
    cb.set_label('m/s')
    cb.ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2g'))
    return cb


def add_ri_cbar(fig, cf, ax_pair, threshold):
    vmax = 5.0 * threshold
    cb   = fig.colorbar(cf, ax=ax_pair, shrink=0.85, pad=0.01)
    cb.set_label('Ri')
    ticks = np.unique(np.round(
        [0, threshold / 2, threshold, 2 * threshold, 5 * threshold], 6))
    cb.set_ticks(ticks[ticks <= vmax + 1e-9])
    cb.ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2g'))
    return cb

## Figure function

In [6]:
def make_figure(U, V, DRHODR, Z_mod, t_mod, lat_label, subfolder, Ri_threshold,
                xlim=None, day_interval=None):
    """
    6-row x 2-col figure for one location and one time window.
    Row 0: raw velocity (UVEL | VVEL)
    Row 1: shear (dU/dz | dV/dz)
    Row 2: Ri = N²/S² (same field both columns)
    Rows 3-5: band-passed 40-60, 25-40, 15-25 days
    """
    dt_h = 3.0
    filt_U = [bpfilt(U, dt_h, bp) for _, bp in BANDS]
    filt_V = [bpfilt(V, dt_h, bp) for _, bp in BANDS]

    dUdz = np.gradient(U, Z_mod, axis=1)
    dVdz = np.gradient(V, Z_mod, axis=1)
    S2   = dUdz ** 2 + dVdz ** 2
    N2   = np.maximum(-(G / RHO0) * DRHODR, 0.0)
    Ri   = np.where(S2 > 1e-12, N2 / S2, np.nan)

    t_arr = t_mod.to_pydatetime()
    title_sfx = 'first 4 wk' if xlim else 'full period'

    fig, axes = plt.subplots(6, 2, figsize=(16, 26),
                             sharex=True, sharey=True,
                             constrained_layout=True)
    fig.suptitle(
        f'TPOSE24 {subfolder} — {LAT_STR[lat_label]}, 140°W  ({title_sfx})',
        fontsize=12, fontweight='bold')

    row_labels = ['Velocity', 'Shear', 'Richardson\nNumber',
                  '40–60 day', '25–40 day', '15–25 day']

    # Row 0: raw velocity
    vm0 = vmax_of(U, V)
    cf_plot(axes[0, 0], t_arr, Z_mod, U, vm0)
    cf0v = cf_plot(axes[0, 1], t_arr, Z_mod, V, vm0)
    axes[0, 0].set_title('UVEL  (m/s)')
    axes[0, 1].set_title('VVEL  (m/s)')
    add_vel_cbar(fig, cf0v, [axes[0, 0], axes[0, 1]])

    # Row 1: shear
    vms = vmax_of(dUdz, dVdz)
    cf_plot(axes[1, 0], t_arr, Z_mod, dUdz, vms)
    cf1v = cf_plot(axes[1, 1], t_arr, Z_mod, dVdz, vms)
    axes[1, 0].set_title('dU/dz  (s⁻¹)')
    axes[1, 1].set_title('dV/dz  (s⁻¹)')
    add_vel_cbar(fig, cf1v, [axes[1, 0], axes[1, 1]])

    # Row 2: Richardson number (same field both columns)
    cf2 = ri_plot(axes[2, 0], t_arr, Z_mod, Ri, Ri_threshold)
    ri_plot(axes[2, 1], t_arr, Z_mod, Ri, Ri_threshold)
    axes[2, 0].set_title('Ri = N² / S²')
    axes[2, 1].set_title('Ri = N² / S²')
    add_ri_cbar(fig, cf2, [axes[2, 0], axes[2, 1]], Ri_threshold)

    # Rows 3-5: band-passed
    for bi, (band_label, _) in enumerate(BANDS):
        row = 3 + bi
        vmb = vmax_of(filt_U[bi], filt_V[bi])
        cf_plot(axes[row, 0], t_arr, Z_mod, filt_U[bi], vmb)
        cfv = cf_plot(axes[row, 1], t_arr, Z_mod, filt_V[bi], vmb)
        axes[row, 0].set_title(f'UVEL  {band_label}')
        axes[row, 1].set_title(f'VVEL  {band_label}')
        add_vel_cbar(fig, cfv, [axes[row, 0], axes[row, 1]])

    # Axis formatting
    for r in range(6):
        for c in range(2):
            depth_yax(axes[r, c])
            if xlim:
                axes[r, c].set_xlim(xlim)
        axes[r, 0].text(-0.13, 0.5, row_labels[r],
                        transform=axes[r, 0].transAxes,
                        rotation=90, va='center', ha='right',
                        fontsize=10, fontweight='bold')

    for c in range(2):
        fmt_time_ax(axes[-1, c], day_interval=day_interval)

    return fig

## Main loop — generate all figures

In [7]:
FOUR_WEEKS = pd.Timedelta(days=28)

for subfolder in SUBFOLDERS:
    Ri_thr   = RI_THRESHOLDS[subfolder]
    save_dir = os.path.join(NOTEBOOK_DIR, subfolder)
    os.makedirs(save_dir, exist_ok=True)

    print(f'\n{"="*60}')
    print(f'{subfolder}  (Ri threshold = {Ri_thr})')
    print(f'{"="*60}')

    try:
        data, Z_mod, t_mod = load_cache(subfolder)
    except FileNotFoundError as exc:
        print(f'  SKIPPED: {exc}')
        continue

    t_start  = t_mod[0]
    xlim_4wk = [t_start.to_pydatetime(),
                (t_start + FOUR_WEEKS).to_pydatetime()]

    fig_num = 1
    for lat_label in LAT_LABELS:
        U      = data[lat_label]['U']
        V      = data[lat_label]['V']
        DRHODR = data[lat_label]['DRHODR']

        # Full time series
        print(f'  fig{fig_num}: {lat_label} 140W full …', end=' ', flush=True)
        fig = make_figure(U, V, DRHODR, Z_mod, t_mod,
                          lat_label, subfolder, Ri_thr)
        out = os.path.join(save_dir, f'fig{fig_num}_{lat_label}_140W_full.png')
        fig.savefig(out, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print('saved')
        fig_num += 1

        # First 4 weeks
        print(f'  fig{fig_num}: {lat_label} 140W 4wk …', end=' ', flush=True)
        fig = make_figure(U, V, DRHODR, Z_mod, t_mod,
                          lat_label, subfolder, Ri_thr,
                          xlim=xlim_4wk, day_interval=4)
        out = os.path.join(save_dir, f'fig{fig_num}_{lat_label}_140W_4wk.png')
        fig.savefig(out, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print('saved')
        fig_num += 1

    print(f'  → {fig_num - 1} figures in {save_dir}/')


3month  (Ri threshold = 0.7)


  Loaded 3month: 727 steps, 2012-10-01 – 2012-12-30, 96 levels (-0.5 to -1250.0 m)
  fig1: 1N 140W full … 

saved
  fig2: 1N 140W 4wk … 

saved
  fig3: 0N 140W full … 

saved
  fig4: 0N 140W 4wk … 

...

saved
  fig3: 0N 140W full … 

saved
  fig4: 0N 140W 4wk … 

saved
  fig5: 1S 140W full … 

saved
  fig6: 1S 140W 4wk … 

saved
  → 6 figures in /home/edavenport/analysis/tpose24-validation/EDJ_interaction/3month_Ri3/

3month_Ri5  (Ri threshold = 0.5)
  SKIPPED: Cache not found: /home/edavenport/analysis/tpose24-validation/EDJ_interaction/3month_Ri5/cache_edj.nc
Run: conda run -n tpose python build_edj_cache.py 3month_Ri5
